In [0]:
#1: Imports & φόρτωση πραγματικών δεδομένων από το Online dataset
import random
import json
import time
from datetime import datetime, timedelta

existing_product_ids = [row["Product ID"] for row in spark.table("dbacademy.Superstore.sales_online_raw").select("Product ID").distinct().collect()]
existing_postal_codes = [row["Postal Code"] for row in spark.table("dbacademy.Superstore.sales_online_raw").select("Postal Code").distinct().collect()]

customer_rows = spark.table("dbacademy.Superstore.sales_online_raw").select("Customer ID", "Customer Name").distinct().collect()
existing_customer_names = {row["Customer ID"]: row["Customer Name"] for row in customer_rows}
existing_customer_ids = list(existing_customer_names.keys())

date_range_row = spark.table("dbacademy.Superstore.sales_online_raw") \
    .selectExpr("min(`Order Date`) as min_date", "max(`Order Date`) as max_date") \
    .collect()[0]

MIN_DATE = datetime.combine(date_range_row["min_date"], datetime.min.time())
MAX_DATE = datetime.combine(date_range_row["max_date"], datetime.min.time())

print(f"Εύρος ημερομηνιών: {MIN_DATE} έως {MAX_DATE}")

print(f"{len(existing_product_ids)} products, {len(existing_postal_codes)} postal codes, {len(existing_customer_ids)} customers")

In [0]:
#2: Fake data generation functions
first_names = ["John","Maria","Wei","Ahmed","Sofia","David","Emma","Lucas","Anna","Ravi"]
last_names = ["Smith","Garcia","Chen","Hassan","Rossi","Kim","Johnson","Silva","Kowalski","Patel"]

def fake_customer():
    if random.random() < 0.3:
        cid = random.choice(existing_customer_ids)
        cname = existing_customer_names[cid]
    else:
        cid = f"POS-{random.randint(10000,99999)}"
        cname = f"{random.choice(first_names)} {random.choice(last_names)}"
    return cid, cname

def random_timestamp_in_range():
    delta = MAX_DATE - MIN_DATE
    random_days = random.randint(0, delta.days)
    random_seconds = random.randint(0, 86400)
    return MIN_DATE + timedelta(days=random_days, seconds=random_seconds)

def fake_pos_transaction():
    cid, cname = fake_customer()
    return {
        "order_id": f"POS-{random.randint(100000,999999)}",
        "customer_id": cid,
        "customer_name": cname,
        "product_id": random.choice(existing_product_ids),
        "postal_code": int(random.choice(existing_postal_codes)),
        "channel": "POS",
        "sale_timestamp": random_timestamp_in_range().isoformat(),
        "quantity": random.randint(1,5),
        "sales": round(random.uniform(10,500), 2),
        "discount": round(random.choice([0,0.1,0.2,0.3]), 2),
    }

fake_pos_transaction()

In [0]:
#3: Batch generation + upload στο Volume
def generate_and_upload_batch(batch_num, size=100):
    batch = [fake_pos_transaction() for _ in range(size)]
    dst = f"/Volumes/dbacademy/Superstore/pos_raw/pos_live_{batch_num:03d}.json"
    with open(dst, "w") as f:
        for record in batch:
            f.write(json.dumps(record) + "\n")
    print(f"Batch {batch_num}: {size} νέες εγγραφές ανέβηκαν")

In [0]:
#4: Loop - παράγει Ν batches, με interval
NUM_BATCHES = 10
INTERVAL_SECONDS = 30

for batch_num in range(NUM_BATCHES):
    generate_and_upload_batch(batch_num)
    time.sleep(INTERVAL_SECONDS)

print("Ολοκληρώθηκε η παραγωγή batches.")

In [0]:
dbutils.fs.head("/Volumes/dbacademy/Superstore/pos_raw/pos_live_000.json")